# Shikibu 速度計測（Colab 本番用）

このノートブックは、**Google Colab の L4 ランタイム**で特徴抽出の速度を測るためのものです。
合格条件は `perf/perf_check.py --profile colab`（1スレッドの純粋計算 20万局面/秒以上、
論理コア数 − 1 スレッドのローダー 100万局面/秒以上）で判定します。

使う前に次を確認してください。
- ランタイムは **GPU（L4）** を選ぶ（CPU の数は GPU ランタイムに依存します。実行時に表示されます）。
- Google Drive に**リポジトリ**があるか、無ければ下の `REPO_URL` から取得します（場所は設定セルの `REPO`）。
- 学習用の PSV があれば、設定セルの `DATA_GLOB` で指定します（無ければ「コピー速度の計測」は飛ばします）。

## 1. Google Drive をマウントする

Drive に保存した wheel（ビルド済みの `shikibu_feat`）と、リポジトリ、データを読むために Drive を使います。
初回は認証画面が出るので、表示に従って許可してください。

In [ ]:
# Google Drive を /content/drive にマウントする（Colab でだけ使える）
from google.colab import drive
drive.mount('/content/drive')

## 2. リポジトリとツールの場所を決める

- `REPO`：リポジトリの場所。Drive にあればそれを使い、無ければ GitHub から取得します（ブランチは `BRANCH`）。
- `WHEEL_DIR`：wheel を保存する Drive の場所。2回目以降の実行はここから入れるので、Rust のビルドを省けます。

In [ ]:
import os
import subprocess
import glob

REPO_URL = "https://github.com/<GitHub のユーザー名>/Shikibu.git"  # 取得元（Drive に無いときだけ使う）
BRANCH = "main"                                             # 計測するブランチ
REPO = "/content/drive/MyDrive/<リポジトリを置くフォルダ>/Shikibu"   # Drive 上のリポジトリ
WHEEL_DIR = "/content/drive/MyDrive/<wheel を置くフォルダ>"         # wheel の保存先
DATA_GLOB = "/content/drive/MyDrive/<学習データのフォルダ>/*.psv"    # 学習用の PSV（任意）

# リポジトリが無ければ取得する
if not os.path.exists(os.path.join(REPO, "perf", "perf_check.py")):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
os.makedirs(WHEEL_DIR, exist_ok=True)

# ランタイムの CPU（論理コア数）を表示する。合格基準のスレッド数はここから決まる
print("論理コア数:", os.cpu_count())
# 目印（<…>）の確認。残っていたら、上の設定を書き換えてから実行する
_settings = {"REPO_URL": REPO_URL, "REPO": REPO, "WHEEL_DIR": WHEEL_DIR, "DATA_GLOB": DATA_GLOB}
_left = [k for k, v in _settings.items() if "<" in v and ">" in v]
if _left:
    raise ValueError("設定セルの <…> を書き換えてください: " + ", ".join(_left))


## 3. wheel をインストールする（無ければビルドして Drive に保存する）

- Drive に `shk_feat-*.whl` があれば、それを入れるだけで済みます（数秒）。
- 無ければ Rust をインストールし、`maturin build --release` で wheel を作って Drive に保存します
  （初回のみ。ソースが変わったら `WHEEL_DIR` の wheel を消して作り直してください）。

In [ ]:
# 既存の wheel を探す。複数あれば一番新しいものを使う
wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)

if wheels:
    # 保存済みの wheel を入れる（ビルドは不要）
    subprocess.run(["pip", "install", "-q", "--force-reinstall", wheels[-1]], check=True)
    print("保存済みの wheel を入れました:", wheels[-1])
else:
    # Rust が無ければ入れる（ランタイムごとに一度だけ必要）
    if subprocess.run(["bash", "-lc", "command -v cargo"], capture_output=True).returncode != 0:
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
    cargo_env = os.path.expanduser("~/.cargo/bin")
    env = dict(os.environ, PATH=cargo_env + os.pathsep + os.environ["PATH"])

    # wheel をビルドして Drive に保存し、そのまま入れる
    subprocess.run(["pip", "install", "-q", "maturin"], check=True)
    subprocess.run(["maturin", "build", "--release", "-m", "crates/shk_feat/Cargo.toml",
                    "-o", WHEEL_DIR], cwd=REPO, env=env, check=True)
    built = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
    subprocess.run(["pip", "install", "-q", "--force-reinstall", built[-1]], check=True)
    print("wheel をビルドして Drive に保存しました:", built[-1])

## 4. テストを流す（正しさの確認）

速度を測る前に、正しさのテストが全件通ることを確認します。合格条件は **全件 passed** です。
通らない場合は、速度の結果を信用しないでください。

In [ ]:
# 正しさのテストを流す。出力の末尾に「passed」（失敗なし）と出れば OK
subprocess.run(["python", "-m", "pytest", "-q", "tests"], cwd=REPO, check=True)

## 5. 速度を測る（合格判定）

`perf/perf_check.py --profile colab` を実行します。純粋計算（1スレッド）とローダー（論理コア数 − 1 スレッド）の
局面/秒を表示し、最後に「総合: PASS」か「総合: FAIL」を出します。
結果の数字は報告に書いてください（環境の CPU と論理コア数も一緒に）。

In [ ]:
# 速度の判定。終了コードは PASS で 0、FAIL で 1
result = subprocess.run(["python", "perf/perf_check.py", "--profile", "colab"], cwd=REPO)
print("終了コード:", result.returncode)

## 6. コピー速度の計測（任意）

学習データのシャードを Drive から `/content/shk_cache` へコピーしながら読み、
ファイルごとのコピー速度（MB/秒）を表示します。毎秒100万局面には約 40MB/秒が必要です（設計書 §S3.2）。
シャードが無ければ、このセルは何も表示せずに終わります。

In [ ]:
import shutil
import shikibu_feat

shards = sorted(glob.glob(DATA_GLOB))
if not shards:
    print("シャードが見つかりません:", DATA_GLOB)
else:
    cache = "/content/shk_cache"
    shutil.rmtree(cache, ignore_errors=True)  # 専用のフォルダから始める
    loader = shikibu_feat.BatchLoader(shards, 16384, 1, num_threads=3,
                                      cache_dir=cache, max_cached=3)
    n = 0
    for batch in loader:          # 最後まで読む（コピーの記録を集めるため）
        n += len(batch["score"])
    print("読んだ局面数:", n)
    for s in loader.cache_stats():
        print(f"{s['file']}: {s['bytes']:,} バイト, {s['seconds']:.2f} 秒, {s['mb_per_s']:.1f} MB/秒")